# mIF Pipeline Debug Notebook

This notebook is a notebook-first wrapper around the Python API in `src/mif_pipeline`.

Use it to:
- inspect resolved config paths
- dry-run the full slide plan
- run one stage at a time and rerun only the failing stage
- inspect outputs between steps

Real data paths are expected to be cluster paths, so the dry-run cells are useful locally.

This notebook assumes `mif_pipeline` is already installed in the active Python environment, for example with `pip install -e .`.

Important setup note: `setup_slide(...)` writes a starter channel map to `setup.channel_map_output`. If you want downstream steps in this notebook session to use that generated file immediately, flip `USE_GENERATED_CHANNEL_MAP = True` below and rerun the config-loading cell after the setup cell has written the file.


In [1]:
from pathlib import Path
import json
import traceback


In [2]:
print("Imports will use the installed mif_pipeline package from the active environment.")


Imports will use the installed mif_pipeline package from the active environment.


In [3]:
from mif_pipeline import (
    build_spatialdata,
    diagnose_label_overlap_instances,
    finalize_nimbus_multislide,
    load_config,
    merge_slide_ometiffs,
    qc_slide,
    run_all,
    run_instanseg,
    run_nimbus_chunked,
    run_nimbus_multislide,
    setup_slide,
)
from mif_pipeline.config import (
    get_slide_config,
    resolve_block_aliases,
    resolve_channel_entries,
    resolve_nimbus_inputs,
    resolve_nimbus_multislide_inputs,
    resolve_spatialdata_channel_entries,
)


In [4]:
# Edit this to the config you want to run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-fullslide.yaml").expanduser()
SLIDE_ID = "SLIDE-0329"
MULTISLIDE_SLIDE_IDS = ["SLIDE-0329"]
RUN_MULTISLIDE_NIMBUS = False
NIMBUS_CHUNK_INDICES = None
RUN_NIMBUS_FINALIZE = True
USE_GENERATED_CHANNEL_MAP = True
RUN_DRY = False
FORCE = False


In [5]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{label} failed: {exc}")
        traceback.print_exc()
        raise


In [6]:
print(f"Config path: {CONFIG_PATH}")
print(f"Config exists: {CONFIG_PATH.exists()}")


Config path: /home/ratnayn/codex/mIF-pipeline/prototyping/prototype_v2-fullslide.yaml
Config exists: True


In [7]:
config = load_config(CONFIG_PATH)

runtime_slide_ids = MULTISLIDE_SLIDE_IDS if RUN_MULTISLIDE_NIMBUS else [SLIDE_ID]
for target_slide_id in runtime_slide_ids:
    target_nimbus = config["slides"].setdefault(target_slide_id, {}).setdefault("nimbus", {})
    target_multislide = target_nimbus.setdefault("multislide", {})
    target_multislide["enabled"] = RUN_MULTISLIDE_NIMBUS
    if RUN_MULTISLIDE_NIMBUS:
        target_multislide["slide_ids"] = list(MULTISLIDE_SLIDE_IDS)

if USE_GENERATED_CHANNEL_MAP:
    for target_slide_id in MULTISLIDE_SLIDE_IDS:
        target_slide = get_slide_config(config, target_slide_id)
        generated_map = target_slide.get("setup", {}).get("channel_map_output")
        if not generated_map:
            raise ValueError(
                f"USE_GENERATED_CHANNEL_MAP is True but setup.channel_map_output is missing for {target_slide_id}."
            )
        config["slides"][target_slide_id]["channel_map_file"] = generated_map

slide = get_slide_config(config, SLIDE_ID)
nimbus_inputs = resolve_nimbus_inputs(config, SLIDE_ID)
if RUN_MULTISLIDE_NIMBUS:
    multislide_nimbus_inputs = resolve_nimbus_multislide_inputs(config, MULTISLIDE_SLIDE_IDS)
else:
    multislide_nimbus_inputs = {
        "slide_ids": [],
        "fov_paths": [],
        "output_dir": None,
        "nimbus_channels": [],
    }
seg_aliases = resolve_block_aliases(
    config,
    SLIDE_ID,
    slide.get("seg_merge", {}) or {},
    block_name="Merge block",
    require_selection=False,
)
full_aliases = resolve_block_aliases(
    config,
    SLIDE_ID,
    slide.get("full_merge", {}) or {},
    block_name="Merge block",
    require_selection=False,
)
seg_entries = resolve_channel_entries(config, SLIDE_ID, seg_aliases)
full_entries = resolve_channel_entries(config, SLIDE_ID, full_aliases)
spatialdata_entries = resolve_spatialdata_channel_entries(config, SLIDE_ID)
cell_mask_path = Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['suffix']}"
nuclear_mask_path = Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['nuclear_suffix']}"
spatialdata_store_path = slide.get("spatialdata", {}).get("store_path")

summary = {
    "slide_id": SLIDE_ID,
    "slide_dir": slide["slide_dir"],
    "output_dir": slide["output_dir"],
    "channel_map_file": slide["channel_map_file"],
    "setup_channel_map_output": slide.get("setup", {}).get("channel_map_output"),
    "seg_merge_inputs": [entry["path"] for entry in seg_entries],
    "full_merge_inputs": [entry["path"] for entry in full_entries],
    "nimbus_raw_paths": nimbus_inputs["raw_paths"],
    "nimbus_fov_paths": nimbus_inputs["fov_paths"],
    "multislide_slide_ids": multislide_nimbus_inputs["slide_ids"],
    "multislide_fov_paths": multislide_nimbus_inputs["fov_paths"],
    "multislide_output_dir": multislide_nimbus_inputs["output_dir"],
    "multislide_nimbus_channels": multislide_nimbus_inputs["nimbus_channels"],
    "spatialdata_aliases": [entry["alias"] for entry in spatialdata_entries],
    "spatialdata_store_path": spatialdata_store_path,
    "segmentation_ome_path": slide["seg_merge"]["ome_path"],
    "cell_mask_path": str(cell_mask_path),
    "nuclear_mask_path": str(nuclear_mask_path),
    "nimbus_chunk_indices": NIMBUS_CHUNK_INDICES,
}
show(summary)


{
  "slide_id": "SLIDE-0329",
  "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329",
  "output_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs",
  "channel_map_file": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
  "setup_channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
  "seg_merge_inputs": [
    "/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/SLIDE-0329_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    "/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/SLIDE-0329_3.0.1_R000_FITC_AF_I.ome.tif",
    "/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F.ome.tif",
    "/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0329/SLIDE-0329_4.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F.ome.tif"
  ],
  "full_merge_inputs": [
    "/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_

In [8]:
def import_status(module_name):
    try:
        __import__(module_name)
        return "OK"
    except Exception as exc:
        return f"MISSING: {exc}"


dependency_status = {
    "yaml": import_status("yaml"),
    "tifffile": import_status("tifffile"),
    "ome_types": import_status("ome_types"),
    "numpy": import_status("numpy"),
    "skimage": import_status("skimage"),
    "instanseg": import_status("instanseg"),
    "tiffslide": import_status("tiffslide"),
    "anndata": import_status("anndata"),
    "sopa": import_status("sopa"),
    "spatialdata": import_status("spatialdata"),
    "shapely": import_status("shapely"),
}

dependency_status


{'yaml': 'OK',
 'tifffile': 'OK',
 'ome_types': 'OK',
 'numpy': 'OK',
 'skimage': 'OK',
 'instanseg': 'OK',
 'tiffslide': 'OK',
 'anndata': "MISSING: No module named 'anndata'",
 'sopa': "MISSING: No module named 'sopa'",
 'spatialdata': "MISSING: No module named 'spatialdata'",
 'shapely': "MISSING: No module named 'shapely'"}

## Dry-Run Planning

Leave `RUN_DRY = True` while you validate config resolution, output paths, and chunk planning.


In [10]:
dry_run_plan = stage_call("run_all(dry_run=True)", run_all, config, SLIDE_ID, dry_run=True)

if RUN_MULTISLIDE_NIMBUS:
    multislide_nimbus_dry_run = stage_call(
        "run_nimbus_multislide(dry_run=True)",
        run_nimbus_multislide,
        config,
        MULTISLIDE_SLIDE_IDS,
        chunk_indices=NIMBUS_CHUNK_INDICES,
        dry_run=True,
    )
else:
    multislide_nimbus_dry_run = {
        "status": "skipped",
        "reason": "Set RUN_MULTISLIDE_NIMBUS = True to preview combined-slide Nimbus execution.",
    }


=== run_all(dry_run=True) ===
{
  "slide_id": "SLIDE-0329",
  "dry_run": true,
  "setup": {
    "slide_id": "SLIDE-0329",
    "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329",
    "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
    "channel_patterns": [
      "*.tif",
      "*.tiff",
      "*.ome.tif",
      "*.ome.tiff"
    ],
    "include_round_in_alias": true,
    "channel_count": 24,
    "aliases": [
      "R1_CY3_AF",
      "R1_CY5_AF",
      "R1_CY7_AF",
      "R1_FITC_AF",
      "R1_BIT2_RS0584_CY3B",
      "R1_BIT3_RS0015_CY5",
      "R1_BIT4_RS0083_750",
      "R1_DAPI",
      "R1_BIT1_RS0996_488",
      "R2_BIT6_RS0639_CY3B",
      "R2_BIT7_RS0109_CY5",
      "R2_BIT8_RS0255_750",
      "R2_DAPI",
      "R2_BIT5_RS1047_488",
      "R3_BIT10_RS0763_CY3B",
      "R3_BIT11_RS1312_CY5",
      "R3_BIT12_RS0237_750",
      "R3_DAPI",
      "R3_BIT9_RS0805_488",
      "R4_P19_POLYRAT",
      "R4_ANXA10_647",
      "R4_LGA

## Run Individual Stages

Set `RUN_DRY = False` above when you are on the cluster and ready to execute a real step.

When `RUN_MULTISLIDE_NIMBUS = True`, the pre-Nimbus stages below run in a simple loop over `MULTISLIDE_SLIDE_IDS`, then Nimbus runs once across that combined slide set. SpatialData and QC remain per-slide.


In [11]:
stage_slide_ids = MULTISLIDE_SLIDE_IDS if RUN_MULTISLIDE_NIMBUS else [SLIDE_ID]
setup_results = {}

In [11]:
for target_slide_id in stage_slide_ids:
    target_slide = get_slide_config(config, target_slide_id)
    if target_slide.get("setup"):
        setup_results[target_slide_id] = stage_call(
            f"setup_slide({target_slide_id}, dry_run={RUN_DRY})",
            setup_slide,
            config,
            target_slide_id,
            force=FORCE,
            dry_run=RUN_DRY,
        )
    else:
        print(f"No setup block configured for {target_slide_id}.")

setup_result = setup_results.get(SLIDE_ID)

=== setup_slide(SLIDE-0329_crop_2048, dry_run=False) ===
{
  "slide_id": "SLIDE-0329_crop_2048",
  "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
  "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/channel_map.generated.json",
  "channel_patterns": [
    "*.tif",
    "*.tiff",
    "*.ome.tif",
    "*.ome.tiff"
  ],
  "include_round_in_alias": true,
  "channel_count": 24,
  "aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8_RS0255_750",
    "R2_DAPI",
    "R2_BIT5_RS1047_488",
    "R3_BIT10_RS0763_CY3B",
    "R3_BIT11_RS1312_CY5",
    "R3_BIT12_RS0237_750",
    "R3_DAPI",
    "R3_BIT9_RS0805_488",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750",
    "R4_DAPI",
    "R4_GFP_POLY_AF488"
  ],
  "status": "skip

In [12]:
merge_results = {}

for target_slide_id in stage_slide_ids:
    merge_results[target_slide_id] = stage_call(
        f"merge_slide_ometiffs({target_slide_id}, dry_run={RUN_DRY})",
        merge_slide_ometiffs,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=RUN_DRY,
    )

merge_result = merge_results[SLIDE_ID]


=== merge_slide_ometiffs(SLIDE-0329_crop_2048, dry_run=False) ===
[merge] starting seg_merge for SLIDE-0329_crop_2048: 5 channels -> /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/SLIDE-0329_crop_2048_segment_merge.ome.tif
[merge] writing SLIDE-0329_crop_2048_segment_merge.ome.tif: 5 channels, 8 pyramid levels
[merge] channel 1/5 R1_DAPI: read level 0
[merge] channel 1/5 R1_DAPI: rebuilding pyramid
[merge] channel 1/5 R1_DAPI: wrote level 0
[merge] channel 1/5 R1_DAPI: wrote pyramid level 1/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 2/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 3/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 4/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 5/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 6/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 7/7
[merge] channel 2/5 R1_FITC_AF_I: reading level 0
[merge] channel 2/5 R1_FITC_AF_I: rebuilding pyramid
[merge] channel 2/5 R1_FITC_AF_I: wrote level 0
[merge] channel 2/5 

In [12]:
instanseg_results = {}

for target_slide_id in stage_slide_ids:
    instanseg_results[target_slide_id] = stage_call(
        f"run_instanseg({target_slide_id}, dry_run={RUN_DRY})",
        run_instanseg,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=RUN_DRY,
    )

instanseg_result = instanseg_results[SLIDE_ID]


=== run_instanseg(SLIDE-0329, dry_run=False) ===
run_instanseg(SLIDE-0329, dry_run=False) failed: Segmentation merge does not exist: /mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome.tif


Traceback (most recent call last):
  File "/tmp/ipykernel_962010/3572664735.py", line 8, in stage_call
    result = func(*args, **kwargs)
  File "/home/ratnayn/codex/mIF-pipeline/src/mif_pipeline/instanseg_runner.py", line 182, in run_instanseg
    raise FileNotFoundError(f"Segmentation merge does not exist: {ome_path}")
FileNotFoundError: Segmentation merge does not exist: /mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome.tif


FileNotFoundError: Segmentation merge does not exist: /mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome.tif

In [14]:
if RUN_MULTISLIDE_NIMBUS:
    nimbus_result = stage_call(
        f"run_nimbus_multislide(dry_run={RUN_DRY})",
        run_nimbus_multislide,
        config,
        MULTISLIDE_SLIDE_IDS,
        chunk_indices=NIMBUS_CHUNK_INDICES,
        force=FORCE,
        dry_run=RUN_DRY,
    )
else:
    nimbus_result = stage_call(
        f"run_nimbus_chunked(dry_run={RUN_DRY})",
        run_nimbus_chunked,
        config,
        SLIDE_ID,
        force=FORCE,
        dry_run=RUN_DRY,
    )


=== run_nimbus_multislide(dry_run=True) ===
{
  "slide_ids": [
    "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2"
  ],
  "mode": "multislide",
  "fov_paths": [
    "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2"
  ],
  "fov_name_to_slide": {
    "SLIDE-0329_crop_2048": "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2": "SLIDE-0329_crop_2048_2"
  },
  "raw_paths_by_slide": {
    "SLIDE-0329_crop_2048": [
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.4_R000_DAPI__FINAL_F.ome.tif",
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_4.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F.ome.tif",
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F.ome.tif",
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_4.0.4_R000_Cy5_Anxa10-647_FINAL_AFR_F.ome.tif"
    ],
    "SLIDE-0329_crop_2048_2": [
      "/mnt/c/

In [18]:
if RUN_MULTISLIDE_NIMBUS and RUN_NIMBUS_FINALIZE:
    nimbus_finalize_result = stage_call(
        f"finalize_nimbus_multislide(dry_run=False)",
        finalize_nimbus_multislide,
        config,
        MULTISLIDE_SLIDE_IDS,
    )
else:
    nimbus_finalize_result = {
        "status": "skipped",
        "reason": "Set RUN_NIMBUS_FINALIZE = True after all multislide chunk jobs have finished.",
    }

nimbus_finalize_result


=== finalize_nimbus_multislide(dry_run=False) ===
{
  "slide_ids": [
    "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2"
  ],
  "mode": "multislide_finalize",
  "status": "written",
  "chunk_count": 4,
  "merged_csv": "/mnt/c/analysis/Data_prototype/nimbus_multislide/cell_table_full.csv",
  "merged_row_count": 8878,
  "merged_columns": [
    "slide_id",
    "cell_id",
    "fov",
    "R1_DAPI",
    "R4_GFP_POLY_AF488",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647"
  ],
  "join_keys": [
    "slide_id",
    "fov",
    "cell_id"
  ],
  "per_slide_tables": {
    "SLIDE-0329_crop_2048": "/mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv",
    "SLIDE-0329_crop_2048_2": "/mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv"
  }
}


{'slide_ids': ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'],
 'mode': 'multislide_finalize',
 'status': 'written',
 'chunk_count': 4,
 'merged_csv': '/mnt/c/analysis/Data_prototype/nimbus_multislide/cell_table_full.csv',
 'merged_row_count': 8878,
 'merged_columns': ['slide_id',
  'cell_id',
  'fov',
  'R1_DAPI',
  'R4_GFP_POLY_AF488',
  'R4_P19_POLYRAT',
  'R4_ANXA10_647'],
 'join_keys': ['slide_id', 'fov', 'cell_id'],
 'per_slide_tables': {'SLIDE-0329_crop_2048': '/mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv',
  'SLIDE-0329_crop_2048_2': '/mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv'}}

In [31]:
spatialdata_results = {}

for target_slide_id in stage_slide_ids:
    spatialdata_results[target_slide_id] = stage_call(
        f"build_spatialdata({target_slide_id}, dry_run={RUN_DRY})",
        build_spatialdata,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=RUN_DRY,
        return_sdata=False,
    )

spatialdata_result = spatialdata_results[SLIDE_ID]


=== build_spatialdata(SLIDE-0329_crop_2048, dry_run=False) ===
[spatialdata] loading image: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/SLIDE-0329_crop_2048_full_merge.ome.tif
[spatialdata] image_load_seconds completed in 0.29s
[spatialdata] loading masks for SLIDE-0329_crop_2048
[spatialdata] mask_load_seconds completed in 0.08s
[spatialdata] vectorizing labels: ['cell_labels', 'nuclear_labels']
[spatialdata] vectorization_seconds completed in 1.41s
[spatialdata] aggregating vector shapes for ['cell_boundaries', 'nuclear_boundaries']


[INFO] (sopa.aggregation.channels) Aggregating channels intensity over 4439 cells with mode='average'


[########################################] | 100% Completed | 3.88 sms


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/spatialdata/models/models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)
[INFO] (sopa.aggregation.channels) Aggregating channels intensity over 4136 cells with mode='average'


[########################################] | 100% Completed | 2.81 sms
[spatialdata] vector_aggregation_seconds completed in 7.21s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv


...

[########################################] | 100% Completed | 2.88 sms
[spatialdata] vector_aggregation_seconds completed in 7.16s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/spatialdata/models/models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


[spatialdata] nimbus_import_seconds completed in 0.18s
[spatialdata] writing store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/spatialdata/models/models.py:1206: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)
/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


[spatialdata] write_seconds completed in 8.02s
[spatialdata] finished SLIDE-0329_crop_2048_2 in 16.99s
{
  "slide_id": "SLIDE-0329_crop_2048_2",
  "status": "written",
  "dry_run": false,
  "pixel_size_um": 0.325,
  "image_key": "SLIDE-0329_crop_2048_2_full_merge",
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/SLIDE-0329_crop_2048_2_full_merge.ome.tif",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/masks/SLIDE-0329_crop_2048_2_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/masks/SLIDE-0329_crop_2048_2_nuclear.tiff",
  "nimbus_table_path": "/mnt/c/analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr",
  "aggregation_aliases": [
    "R1_DAPI",
    "R4_GFP_POLY_AF488",
    "R4_P19_POLYRAT",
  

In [14]:
slide

{'pixel_size_um': 0.325,
 'setup': {'channel_patterns': ['*.tif', '*.tiff', '*.ome.tif', '*.ome.tiff'],
  'channel_map_output': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/channel_map.generated.json',
  'include_round_in_alias': True},
 'seg_merge': {'enabled': True,
  'channels': ['R1_DAPI',
   'R1_FITC_AF_I',
   'R1_CY3_AF_I',
   'R1_CY5_AF_I',
   'R1_CY7_AF_I'],
  'suffix': '_segment_merge.ome.tif',
  'compression': 'zlib',
  'tile': [256, 256],
  'bigtiff': True,
  'ome_path': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/SLIDE-0329_crop_2048_2_segment_merge.ome.tif'},
 'full_merge': {'enabled': True,
  'exclude_channels': [],
  'suffix': '_full_merge.ome.tif',
  'compression': 'zlib',
  'tile': [256, 256],
  'bigtiff': True,
  'ome_path': '/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/SLIDE-0329_crop_2048_2_full_merge.ome.tif'},
 'instanseg': {'model': 'fluorescence_nuclei_and_cells',
  'mode': 'medium',
  'prediction_tag': '_ins

In [11]:
import spatialdata as sd

sdata = sd.read_zarr(slide["spatialdata"]["store_path"])
sdata


no parent found for <ome_zarr.reader.Label object at 0x728b4045dca0>: None
no parent found for <ome_zarr.reader.Label object at 0x728b40476c30>: None


SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4439, 4) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4136, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_boundaries': AnnData (4439, 24)
      ├── 'agg_nuclear_boundaries': AnnData (4136, 24)
      └── 'nimbus_table': AnnData (4439, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes), nuclear_boundaries (Shapes)

In [ ]:
diagnose_label_overlap_instances(sdata["cell_labels"], sdata["nuclear_labels"])


In [ ]:
config["slides"]


In [ ]:
import numpy as np
import tifffile
from pathlib import Path

cell_mask_path = Path(
    instanseg_result["cell_mask_path"]
    if "instanseg_result" in globals() and "cell_mask_path" in instanseg_result
    else Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['suffix']}"
)
nuclear_mask_path = Path(
    instanseg_result["nuclear_mask_path"]
    if "instanseg_result" in globals() and "nuclear_mask_path" in instanseg_result
    else Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['nuclear_suffix']}"
)

cells = np.asarray(tifffile.imread(cell_mask_path))
nuclear = np.asarray(tifffile.imread(nuclear_mask_path))

diagnose_label_overlap_instances(cells, nuclear)


In [ ]:
qc_result = stage_call("qc_slide", qc_slide, config, SLIDE_ID)


## Full Run Shortcut

After debugging individual stages, you can use the single `run_all(...)` wrapper below for one slide.

`run_all(...)` is still single-slide, so multislide Nimbus should be tested with the staged cells above rather than this shortcut.


In [ ]:
run_all_result = stage_call(
    f"run_all(dry_run={RUN_DRY})",
    run_all,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)
